# 천리안 GK2A 위성 이미지 기반 이상기상 분류
CLIP ViT-L/14 기반 멀티모달 분류 모델 (위성이미지 6장 + ASOS 수치 텍스트)

In [ ]:
# 셀 1: 전체 import + 설정
import torch
import torch.nn as nn
import json
import os
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from tqdm import tqdm
import pandas as pd
from collections import Counter
from google.colab import drive
drive.mount('/content/drive')

WORK = "/content/drive/MyDrive/GK2A_Train"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LABELS = ['호우', '강풍', '대설', '한파', '폭염', '태풍', '정상']
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}
SAVE_DIR = f"{WORK}/output/clip_classifier_v2"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Device: {DEVICE}")

In [ ]:
# 셀 2: 데이터셋
class WeatherDataset(Dataset):
    def __init__(self, json_path, clip_processor, max_text_len=77):
        with open(json_path, 'r') as f:
            self.data = json.load(f)
        self.processor = clip_processor
        self.max_text_len = max_text_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        label = LABEL2ID[sample['conversations'][1]['value']]
        images = [Image.open(p).convert('RGB').resize((224, 224)) for p in sample['images']]
        text = sample['conversations'][0]['value'].replace("<image>" * 6 + "\n", "")
        if "[지상관측 수치]" in text:
            start = text.index("[지상관측 수치]")
            text = text[start:start+400]
        else:
            text = text[:400]
        img_inputs = self.processor(images=images, return_tensors="pt", padding=True)
        txt_inputs = self.processor(
            text=text, return_tensors="pt",
            padding='max_length', max_length=self.max_text_len, truncation=True
        )
        return {
            'pixel_values': img_inputs['pixel_values'],
            'input_ids': txt_inputs['input_ids'].squeeze(0),
            'attention_mask': txt_inputs['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }

def collate_fn(batch):
    return {
        'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
        'input_ids': torch.stack([b['input_ids'] for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'label': torch.stack([b['label'] for b in batch])
    }

In [ ]:
# 셀 3: 모델 정의
class WeatherClassifier(nn.Module):
    def __init__(self, clip_model, num_classes=7, dropout=0.3):
        super().__init__()
        self.clip = clip_model
        hidden = self.clip.config.projection_dim  # 768

        self.classifier = nn.Sequential(
            nn.Linear(hidden * 6 + hidden, 1024),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, pixel_values, input_ids, attention_mask):
        img_features = []
        for i in range(6):
            outputs = self.clip.vision_model(pixel_values[:, i])
            pooled = outputs.pooler_output
            projected = self.clip.visual_projection(pooled)
            img_features.append(projected)
        img_feat = torch.cat(img_features, dim=1)

        txt_outputs = self.clip.text_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        txt_feat = self.clip.text_projection(txt_outputs.pooler_output)
        combined = torch.cat([img_feat, txt_feat], dim=1)
        return self.classifier(combined)

In [ ]:
# 셀 4: 모델 로드 + 전체 unfreeze
print("CLIP 로드 중...")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")

# 전체 unfreeze (핵심 변경)
for param in clip_model.parameters():
    param.requires_grad = True

model = WeatherClassifier(clip_model).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"학습 파라미터: {trainable:,} / 전체: {total:,} ({100*trainable/total:.2f}%)")

# 데이터로더
train_dataset = WeatherDataset(f"{WORK}/data/train_llama_v5_336.json", clip_processor)
val_dataset   = WeatherDataset(f"{WORK}/data/val_llama_v5_336.json", clip_processor)
test_dataset  = WeatherDataset(f"{WORK}/data/test_llama_v5_336.json", clip_processor)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,
                          collate_fn=collate_fn, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False,
                          collate_fn=collate_fn, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=8, shuffle=False,
                          collate_fn=collate_fn, num_workers=2)

print(f"Train: {len(train_dataset)} / Val: {len(val_dataset)} / Test: {len(test_dataset)}")

In [ ]:
# 셀 5: 학습
EPOCHS = 15
SAVE_DIR = f"{WORK}/output/clip_classifier_v2"

# 파라미터 그룹별 학습률 다르게
# CLIP 인코더: 낮은 lr (사전학습 망가지지 않게)
# FC 헤드: 높은 lr
optimizer = AdamW([
    {'params': model.clip.parameters(), 'lr': 1e-5},
    {'params': model.classifier.parameters(), 'lr': 1e-3}
], weight_decay=0.01)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=len(train_loader),
    num_training_steps=EPOCHS * len(train_loader)
)
criterion = nn.CrossEntropyLoss()
best_val_f1 = 0.0

for epoch in range(EPOCHS):
    model.train()
    train_loss, train_preds, train_labels = 0.0, [], []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} Train"):
        pixel_values   = batch['pixel_values'].to(DEVICE)
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        logits = model(pixel_values, input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        train_loss += loss.item()
        train_preds.extend(logits.argmax(dim=-1).cpu().tolist())
        train_labels.extend(labels.cpu().tolist())

    train_f1 = f1_score(train_labels, train_preds, average='macro')
    train_loss /= len(train_loader)

    model.eval()
    val_preds, val_labels, val_loss = [], [], 0.0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} Val"):
            pixel_values   = batch['pixel_values'].to(DEVICE)
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['label'].to(DEVICE)

            logits = model(pixel_values, input_ids, attention_mask)
            val_loss += criterion(logits, labels).item()
            val_preds.extend(logits.argmax(dim=-1).cpu().tolist())
            val_labels.extend(labels.cpu().tolist())

    val_f1 = f1_score(val_labels, val_preds, average='macro')
    val_loss /= len(val_loader)

    print(f"\nEpoch {epoch+1}: train_loss={train_loss:.4f} train_f1={train_f1:.4f} "
          f"val_loss={val_loss:.4f} val_f1={val_f1:.4f}")
    print(f"  예측 분포: {Counter([ID2LABEL[p] for p in val_preds])}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), f"{SAVE_DIR}/best_model.pt")
        print(f"  ✅ 최고 모델 저장 (val_f1={val_f1:.4f})")

print(f"\n학습 완료! Best Val F1: {best_val_f1:.4f}")

In [ ]:
# 셀 6: 평가 (독립 실행 가능 — 학습 없이 best_model.pt만 로드)
import torch
import torch.nn as nn
import json, os
import numpy as np
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
drive.mount('/content/drive')

# 한글 폰트
import subprocess
subprocess.run(['apt-get', 'install', '-qq', 'fonts-nanum'], capture_output=True)
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

WORK     = "/content/drive/MyDrive/GK2A_Train"
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
LABELS   = ['호우', '강풍', '대설', '한파', '폭염', '태풍', '정상']
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}
SAVE_DIR = f"{WORK}/output/clip_classifier_v2"

class WeatherDataset(Dataset):
    def __init__(self, json_path, clip_processor, max_text_len=77):
        with open(json_path, 'r') as f:
            self.data = json.load(f)
        self.processor = clip_processor
        self.max_text_len = max_text_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        label  = LABEL2ID[sample['conversations'][1]['value']]
        images = [Image.open(p).convert('RGB').resize((224, 224)) for p in sample['images']]
        text   = sample['conversations'][0]['value'].replace("<image>" * 6 + "\n", "")
        if "[지상관측 수치]" in text:
            start = text.index("[지상관측 수치]")
            text  = text[start:start+400]
        else:
            text = text[:400]
        img_inputs = self.processor(images=images, return_tensors="pt", padding=True)
        txt_inputs = self.processor(text=text, return_tensors="pt",
                                    padding='max_length', max_length=self.max_text_len, truncation=True)
        return {
            'pixel_values': img_inputs['pixel_values'],
            'input_ids':    txt_inputs['input_ids'].squeeze(0),
            'attention_mask': txt_inputs['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }

def collate_fn(batch):
    return {
        'pixel_values':   torch.stack([b['pixel_values']   for b in batch]),
        'input_ids':      torch.stack([b['input_ids']      for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'label':          torch.stack([b['label']          for b in batch])
    }

class WeatherClassifier(nn.Module):
    def __init__(self, clip_model, num_classes=7, dropout=0.3):
        super().__init__()
        self.clip = clip_model
        hidden = self.clip.config.projection_dim
        self.classifier = nn.Sequential(
            nn.Linear(hidden * 6 + hidden, 1024), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(1024, 256),                 nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, pixel_values, input_ids, attention_mask):
        img_features = []
        for i in range(6):
            out = self.clip.vision_model(pixel_values[:, i])
            projected = self.clip.visual_projection(out.pooler_output)
            img_features.append(projected)
        img_feat = torch.cat(img_features, dim=1)
        txt_out  = self.clip.text_model(input_ids=input_ids, attention_mask=attention_mask)
        txt_feat = self.clip.text_projection(txt_out.pooler_output)
        return self.classifier(torch.cat([img_feat, txt_feat], dim=1))

print("CLIP 로드 중...")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
clip_model     = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
model          = WeatherClassifier(clip_model).to(DEVICE)

model.load_state_dict(torch.load(f"{SAVE_DIR}/best_model.pt", map_location=DEVICE))
model.eval()
print("best_model.pt 로드 완료")

val_dataset = WeatherDataset(f"{WORK}/data/val_llama_v5_336.json", clip_processor)
val_loader  = DataLoader(val_dataset, batch_size=8, shuffle=False,
                         collate_fn=collate_fn, num_workers=2)
print(f"Val 샘플 수: {len(val_dataset)}")

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(val_loader, desc="Eval"):
        logits = model(batch['pixel_values'].to(DEVICE),
                       batch['input_ids'].to(DEVICE),
                       batch['attention_mask'].to(DEVICE))
        all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
        all_labels.extend(batch['label'].numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

print("\n" + "="*70)
print(classification_report(all_labels, all_preds,
                             target_names=LABELS, digits=4, zero_division=0))

true_dist = Counter(all_labels.tolist())
pred_dist = Counter(all_preds.tolist())
print(f"{'클래스':<8} {'실제':>8} {'예측':>8} {'차이':>8}")
for i, name in enumerate(LABELS):
    t, p = true_dist.get(i, 0), pred_dist.get(i, 0)
    print(f"{name:<8} {t:>8} {p:>8} {p-t:>+8}")

cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.heatmap(cm,      annot=True, fmt='d',   cmap='Blues',
            xticklabels=LABELS, yticklabels=LABELS, ax=axes[0])
axes[0].set_title('Confusion Matrix (절대값)')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1,
            xticklabels=LABELS, yticklabels=LABELS, ax=axes[1])
axes[1].set_title('Confusion Matrix (Recall)')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()

errors = [(LABELS[i], LABELS[j], cm[i][j])
          for i in range(len(LABELS)) for j in range(len(LABELS))
          if i != j and cm[i][j] > 0]
errors.sort(key=lambda x: -x[2])
print("\n가장 많이 틀린 패턴 Top 10")
for tl, pl, cnt in errors[:10]:
    print(f"  {tl:>4} → {pl:<4}: {cnt}건")